# 🎯 [K15_M03] 조건을 넣어 빈 시설 검색하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m03_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 03. 조건을 넣어 빈 시설 검색하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 03. 조건을 넣어 빈 시설 검색하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>시설 예약표만을 근거로 특정 요일과 시간대의 가용 시설을 정확하게 찾아내는 조건 기반 프롬프트를 작성합니다. 단순 질문과 참조 표를 제공한 프롬프트의 검색 결과를 비교 분석하며, 요일·시간대 등 복합 조건 질의 4개를 단계별로 실행합니다. 이를 통해 AI가 자의적 추측 대신 주어진 근거 데이터 내에서만 논리적으로 빈 시설을 탐색하도록 유도하는 질의 기법을 익힙니다.</p>

**[학습 목표 및 체득 역량]**
<p>예약 현황표를 근거로 요일·시간·시설 조건을 결합한 프롬프트를 작성하고 질문 4개를 실행하여 표 유무에 따른 답변 차이를 분석합니다. <strong>이를 통해 AI의 환각을 방지하고 명확한 참조 데이터를 기반으로 원하는 결과를 정확히 도출하는 조건 제어 프롬프트 엔지니어링 기법을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 조건 검색 프롬프트를 쓴다(요일·시간·시설).
<h4>이 단계의 역할</h4><p>시설, 요일, 시간대 조건을 명확히 반영하여 AI가 참조 데이터에 기반해서만 검색하도록 지시하는 '조건 검색 기본 프롬프트'를 작성합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>AI에게 '한빛대대 시설 정보 검색 도우미' 역할을 부여하는 페르소나 지시문을 작성합니다.</p></li>
<li><p>'제공된 [예약 현황표]에 명시된 내용만 근거로 답변할 것', '점검·소독 칸은 이용 불가로 안내할 것' 제약조건을 정의합니다.</p></li>
<li><p>조건 검색 템플릿 구조(역할 - 지침 - 근거 데이터 영역 - 질의)를 완성합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 설계 예시</h4><blockquote><p><strong>[프롬프트 템플릿 질의]:</strong> "너는 가상 한빛대대 시설 정보 검색 도우미다.<br>반드시 아래 [예약 현황표]에 있는 내용만을 근거로 답하라.<br>'점검', '소독' 칸은 빈 칸이 아니라 이용 불가로 취급하라.<br>표에 명시된 내용 외에 임의로 추측하여 지어내지 말라.<br>[예약 현황표]: [표 데이터 입력]<br>[질문]: [검색 조건 입력]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>필수 제약 포함:</strong> 근거 데이터 한정 지침과 점검/소독 제외 규칙이 프롬프트에 명시되었는가?</li>
<li><strong>모호성 배제:</strong> AI가 외부 지식을 활용하지 못하도록 명확히 통제되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>조건 검색 프롬프트 템플릿 완성도</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>작성한 조건 검색 프롬프트 초안</p></li></ul>

#### 🔹 Step 2: 예약표를 붙여 넣는다.
<h4>이 단계의 역할</h4><p>작성한 프롬프트 템플릿의 [예약 현황표] 영역에 미션2에서 정리한 28행 현황표 데이터를 정확히 삽입합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>미션2에서 정제한 28행 예약 현황표 마크다운 텍스트를 복사합니다.</p></li>
<li><p>프롬프트의 데이터 슬롯에 누락된 행이나 깨진 구분자(|) 없이 온전하게 붙여 넣습니다.</p></li>
<li><p>AI가 표의 컬럼(시설, 요일, 5개 시간대)을 정상적으로 파싱하는지 확인합니다.</p></li>
</ol>
<h4>💡 AI 데이터 파싱 확인 질의 예시</h4><blockquote><p><strong>[데이터 수신 점검 질의]:</strong> "[예약 현황표 데이터 입력 후] 위 표에서 대대 강당의 금요일 11-13 일정과 체력단련실 수요일 14-16 일정이 무엇으로 표시되어 있는지 1문장으로 확인해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>데이터 정합성:</strong> 붙여넣기 과정에서 특정 요일이나 시간대가 누락되지 않았는가?</li>
<li><strong>인식 확인:</strong> 점검(음향)과 소독(이용 불가) 텍스트가 정확히 인식되는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>28행 예약 데이터 결합 프롬프트 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>참조 데이터가 결합된 프롬프트 완성본</p></li></ul>

#### 🔹 Step 3: 질문 4개로 검색한다.
<h4>이 단계의 역할</h4><p>질문 카드 1~4번(조건 검색 질의 4개)을 입력하여 AI의 검색 답변을 획득합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>질문 1(수요일 저녁 빈 강당), 질문 2(목요일 오후 세미나실 A, B 빈 곳), 질문 3(금요일 오전 강당 가용 여부), 질문 4(토요일 종일 비는 시설)를 순차 질의합니다.</p></li>
<li><p>AI가 각 질문에 대해 도출한 답변 텍스트를 그대로 기록합니다.</p></li>
<li><p>수요일 저녁(19-21) 강당 가용 여부에 대해 14-16 빈 칸과 혼동하지 않고 정확히 '19-21 이용 가능'으로 답했는지 확인합니다.</p></li>
</ol>
<h4>💡 AI 조건 검색 실행 예시</h4><blockquote><p><strong>[조건 검색 질의 실행]:</strong> "[프롬프트 본문 뒤에 추가]<br>[질문 1]: 수요일 저녁에 빈 강당 있어요?<br>[질문 2]: 목요일 오후(14시 이후)에 세미나실 A, B 중 빈 곳은 어디인가요?"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>질문 1 검증:</strong> 수요일 강당의 빈 시간(14-16, 19-21) 중 '저녁' 조건에 부합하는 19-21시를 짚었는가?</li>
<li><strong>질문 3 검증:</strong> 금요일 11-13 점검(음향)으로 인해 11시 이후 이용 불가함을 반영했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>질문 4개에 대한 AI 원시 응답 결과 4건 확보</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>질문 4개 질의 내역 및 AI 생성 답변 기록</p></li></ul>

#### 🔹 Step 4: 표 없이 같은 질문을 해 비교한다.
<h4>이 단계의 역할</h4><p>참조 예약 현황표를 제공하지 않고 동일한 질문 4개를 던졌을 때의 답변과 비교하여, 데이터 근거 프롬프트의 중요성을 입증합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>새 대화창을 열고 예약표를 전혀 제공하지 않은 채 "수요일 저녁에 빈 강당 있어요?" 등 질문 4개를 동일하게 묻습니다.</p></li>
<li><p>표가 없을 때 AI가 생성하는 환각(가상의 일정 생성, 일반적인 부대 이용 수칙 나열 등)을 캡처합니다.</p></li>
<li><p>표가 있을 때와 없을 때의 응답 정확도 및 실무 유용성을 비교표로 정리합니다.</p></li>
</ol>
<h4>💡 AI 무근거 질의 비교 예시</h4><blockquote><p><strong>[무근거 질의]:</strong> "(참조 데이터 없이) 부대 대대 강당 수요일 저녁 19시부터 21시까지 비어 있어? 예약 가능한지 알려줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>환각 현상 확인:</strong> 표가 없을 때 AI가 존재하지 않는 가상의 부대 규정을 꾸며내거나 임의로 비어 있다고 답하는 오류를 식별했는가?</li>
<li><strong>근거 기반 차이:</strong> 참조 데이터 제공 유무가 신뢰성에 미치는 결정적 차이를 명확히 기록했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>참조 데이터 유무에 따른 AI 응답 비교표 작성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>표 유무 비교 분석표 및 환각 사례 캡처</p></li></ul>

#### 🔹 Step 5: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>시간대 표현('저녁' vs '19-21시')을 정량적 숫자로 바꾸어 질의했을 때 AI 응답의 일관성을 검증하는 심화 도전을 수행합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>'수요일 저녁' 대신 '수요일 19시 이후' 또는 '수요일 19:00~21:00'로 변경하여 재질의합니다.</p></li>
<li><p>모호한 구어 표현과 명확한 24시간 표기법 간 응답의 정확도 차이를 관찰합니다.</p></li>
<li><p>검색 프롬프트에 시간 범위를 명확히 규정하는 가이드가 필요한 이유를 도출합니다.</p></li>
</ol>
<h4>💡 AI 정량 질의 심화 예시</h4><blockquote><p><strong>[정량 조건 질의]:</strong> "[예약표 포함] 수요일 19:00~21:00 시간대에 대대 강당의 예약 상태가 '빈' 상태인지 단답형과 근거로 답변해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>결과 일관성:</strong> 표현 방식을 바꾸어도 동일하게 '19-21시 빈 상태'라는 팩트가 도출되는가?</li>
<li><strong>해석 오류 방지:</strong> '오후'나 '저녁'의 시간 범위를 AI가 자의적으로 왜곡하지 않았는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>정량 조건 질의 수행 결과 및 일관성 확인</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 조건 변경 질의 결과 및 비교 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 조건 검색 프롬프트 초안
```markdown
너는 가상 한빛대대 시설 정보 검색 도우미다.
반드시 아래 [예약 현황표]에 명시된 내용만을 근거로 답하라.
'점검', '소독' 칸은 빈 칸이 아니라 이용 불가로 취급하라.
표에 없는 내용은 임의로 지어내지 말고, 근거 행을 답변에 표기하라.

[예약 현황표]
(28행 마크다운 표 삽입)

[질문]
(질문 입력)
```

## 2. 질문 4개 검색 결과 기록지
| 번호 | 질문 내용 | AI 검색 답변 | 원본 표 일치 여부 |
| :---: | :--- | :--- | :---: |
| 1 | 수요일 저녁에 빈 강당 있어요? | 대대 강당은 수요일 19-21시에 비어 있어 이용 가능합니다. (근거: 대대 강당 | 수 | 19-21 | 빈) | 맞음 (PASS) |
| 2 | 목요일 오후에 세미나실 A, B 중 빈 곳은? | 세미나실 A는 14-16시, 16-18시 모두 이용 가능하며, 세미나실 B는 14-16시에 이용 가능합니다. (16-18시는 AI 활용 스터디 배정) | 맞음 (PASS) |
| 3 | 금요일 오전에 강당 쓸 수 있나요? | 이용 불가합니다. 09-11시는 월례 조회, 11-13시는 음향 점검 일정으로 배정되어 있습니다. | 맞음 (PASS) |
| 4 | 토요일에 하루 종일 비는 시설이 있나요? | 하루 종일 비어 있는 시설은 없습니다. 각 시설마다 최소 1건 이상 일정이 배정되어 있습니다. | 맞음 (PASS) |

## 3. 참조 표 유무에 따른 답변 비교 (환각 분석)
| 구분 | 표를 제공했을 때 | 표를 제공하지 않았을 때 (무근거 질의) |
| :--- | :--- | :--- |
| 답변 양상 | 19-21시 빈 상태라고 데이터 기반 답변 | "일반적으로 부대 강당은 저녁 점호 전 비어 있으니..." 환각/추측 답변 |
| 정확성 판정 | PASS (원문 일치) | FAIL (허위 정보 생성) |

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 조건 제어 프롬프트 작성 — 참조 데이터를 결합하여 조건 검색 프롬프트를 설계하고 데이터 유무에 따른 환각 차이를 입증했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>근거 표 결합 및 제약조건이 명시된 조건 검색 프롬프트 완성</p></li><li><p>질문 4개 질의 결과 도출 및 수요일 저녁(19-21) 가용 여부 정확 판별</p></li><li><p>참조 데이터 유무에 따른 답변 차이 및 환각 분석표 증빙</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>AI가 외부 지식을 상상해서 답하지 못하도록 '표 내용에만 근거할 것' 제약조건을 올바르게 설정했는지 확인한다.</p></blockquote>


In [ ]:
# [미션 03] 프롬프트 및 실습 결과 기록
mission_03_prompt = """# [미션 03] 조건 검색 기본 프롬프트 설계 및 질의 실행
너는 가상 한빛대대 시설 정보 검색 도우미다.
반드시 아래 [예약 현황표]에 명시된 내용만을 근거로 답하라.
'점검', '소독' 칸은 빈 칸이 아니라 이용 불가로 취급하라.
표에 없는 내용은 임의로 지어내지 말고, 근거 행을 답변에 표기하라.

[예약 현황표]
(4개 시설 28행 1주 예약 현황 마크다운 표 입력)

[질문]
1. 수요일 저녁에 빈 강당 있어요?
2. 목요일 오후에 세미나실 A, B 중 빈 곳은?
3. 금요일 오전에 강당 쓸 수 있나요?
4. 토요일에 하루 종일 비는 시설이 있나요?
"""

mission_03_result = """### 📋 조건 검색 질의 결과 및 환각 비교 리포트

#### 1. 질문 4개 검색 결과 기록지
| 번호 | 질문 내용 | AI 검색 답변 핵심 | 원본 표 일치 여부 |
| :---: | :--- | :--- | :---: |
| 1 | 수요일 저녁에 빈 강당 있어요? | 대대 강당은 수요일 19-21시에 비어 있어 이용 가능합니다. (근거: 대대 강당 \| 수 \| 19-21 \| 빈) | 맞음 (PASS) |
| 2 | 목요일 오후에 세미나실 A, B 중 빈 곳은? | 세미나실 A는 14-16시, 16-18시 모두 이용 가능하며, 세미나실 B는 14-16시에 이용 가능합니다. (16-18시는 AI 활용 스터디 배정) | 맞음 (PASS) |
| 3 | 금요일 오전에 강당 쓸 수 있나요? | 이용 불가합니다. 09-11시는 월례 조회, 11-13시는 음향 점검 일정으로 배정되어 있습니다. | 맞음 (PASS) |
| 4 | 토요일에 하루 종일 비는 시설이 있나요? | 하루 종일 비어 있는 시설은 없습니다. 각 시설마다 최소 1건 이상 일정이 배정되어 있습니다. | 맞음 (PASS) |

#### 2. 참조 표 유무에 따른 답변 비교 (환각 분석)
| 구분 | 표를 제공했을 때 | 표를 제공하지 않았을 때 (무근거 질의) |
| :--- | :--- | :--- |
| 답변 양상 | 19-21시 빈 상태라고 데이터 기반 답변 | "일반적으로 부대 강당은 저녁 점호 전 비어 있으니..." 환각/추측 답변 |
| 정확성 판정 | **PASS (원문 일치)** | **FAIL (허위 정보 생성)** |
"""

print(f"미션 03 실습 기록 완료: 프롬프트 {len(mission_03_prompt.strip())}자, 결과 {len(mission_03_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션3] 수요일 강당 가용 슬롯 필터링
wed_auditorium = df_schedule[(df_schedule['facility'] == '대대 강당') & (df_schedule['day'] == '수')]
print("=== 수요일 대대 강당 일정 ===")
for c in time_cols:
    status = wed_auditorium[c].values[0]
    print(f"- {c} : {status}")

evening_status = wed_auditorium['19-21'].values[0]
print(f"\n검증 결과: 저녁(19-21) 강당 상태는 '{evening_status}' 입니다.")
